# 00 2-site DMRG：1回の局所更新を追う穴埋め演習

小さい3-site実数MPSを使い、2 coreの結合から局所有効Hamiltonian、固有値問題、SVD分割、bond dimension制限までをPyTorchで確認します。

このNotebookの対象は**1回の2-site局所更新**です。sweep全体、MPO、NNへの適用は扱いません。

## 1. 学習目的と進め方

次の操作を区別して実装します。

1. **結合**：2 coreを1つの中心テンソルへまとめる。状態は変えない。
2. **局所最適化**：有効Hamiltonianの最小固有ベクトルを中心へ入れる。状態とエネルギーが変わり得る。
3. **打ち切りなしのSVD分割**：更新後の中心を2 coreへ戻す。状態は変えない。
4. **SVD打ち切り**：bond dimensionを制限する。状態とエネルギーが変わり得る。

コード中の`___`が穴埋め箇所です。各関数の`Args`と`Returns`、直前の数式、検証条件から実装してください。解答は掲載していません。

冒頭で`___ = None`としているため、未回答の段階は検証をスキップし、Notebook全体をRun Allできます。実装した段階だけ検証が実行されます。

In [14]:
from __future__ import annotations
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import math

import torch

DEVICE = torch.device("cpu")
DTYPE = torch.float64
ATOL = 1e-10
RTOL = 1e-10

# 穴埋め前でもNotebook全体を実行できるようにする未実装値。
# 各TODOでは右辺の___をPyTorchコードへ置き換える。
___ = None

print("device / dtype:", DEVICE, DTYPE)

device / dtype: cpu torch.float64


## 2. 共通の数値例

初期MPSは

$$
G_1,G_2,G_3\in\mathbb R^{1\times2\times1}
$$

で、値を

$$
(G_1)_{1,:,1}=(1,0),\qquad
(G_2)_{1,:,1}=(1,2),\qquad
(G_3)_{1,:,1}=(3,4)
$$

とします。第1coreを固定し、第2・第3coreを更新します。

中心のベクトル化順序は

$$
(i_2,i_3)=(1,1),(1,2),(2,1),(2,2),
$$

全状態は

$$
(i_1,i_2,i_3)=
(1,1,1),(1,1,2),(1,2,1),(1,2,2),
(2,1,1),(2,1,2),(2,2,1),(2,2,2)
$$

の順に並べます。これはPyTorchの通常の`reshape(-1)`と整合します。

In [15]:
G1 = torch.tensor([[[1.0], [0.0]]], dtype=DTYPE, device=DEVICE)
G2 = torch.tensor([[[1.0], [2.0]]], dtype=DTYPE, device=DEVICE)
G3 = torch.tensor([[[3.0], [4.0]]], dtype=DTYPE, device=DEVICE)

h = torch.tensor(
    [
        [-1.0, 0.0, 0.0, -1.0],
        [0.0, 1.0, -1.0, 0.0],
        [0.0, -1.0, 1.0, 0.0],
        [-1.0, 0.0, 0.0, -1.0],
    ],
    dtype=DTYPE,
    device=DEVICE,
)

assert G1.shape == G2.shape == G3.shape == (1, 2, 1)
assert h.shape == (4, 4)
torch.testing.assert_close(h, h.T, atol=ATOL, rtol=RTOL)
print("initial tensors: OK")

initial tensors: OK


## 3. 第1段階：2 coreの結合と全状態の再構成

### 目的

$G_2,G_3$を結合して中心テンソル$B_{\mathrm{old}}$を作り、結合前後で全状態が変わらないことを確認します。

$$
(B_{\mathrm{old}})_{\alpha i_2i_3 1}
=
\sum_\beta
(G_2)_{\alpha i_2\beta}(G_3)_{\beta i_3 1}
$$

### 期待するshapeと値

$$
B_{\mathrm{old}}\in\mathbb R^{1\times2\times2\times1},\qquad
(B_{\mathrm{old}})_{1,:,:,1}=
\begin{pmatrix}3&4\\6&8\end{pmatrix}.
$$

`merge_two_cores`、`contract_three_cores`、`contract_fixed_left`の縮約式を実装してください。

In [16]:
def merge_two_cores(
    left_core: torch.Tensor,
    right_core: torch.Tensor,
) -> torch.Tensor | None:
    """隣接する2 coreの共有bondを縮約する。

    Args:
        left_core: shape (r_left, n_left, r_middle) の左core。
        right_core: shape (r_middle, n_right, r_right) の右core。

    Returns:
        shape (r_left, n_left, n_right, r_right) の結合テンソル。
    """

    return torch.einsum("ijk,krl->ijrl",left_core,right_core)


def contract_three_cores(
    core1: torch.Tensor,
    core2: torch.Tensor,
    core3: torch.Tensor,
) -> torch.Tensor | None:
    """3 coreを縮約して全状態を作る。

    Args:
        core1: shape (1, n1, r1) の第1core。
        core2: shape (r1, n2, r2) の第2core。
        core3: shape (r2, n3, 1) の第3core。

    Returns:
        shape (n1, n2, n3) の全状態テンソル。
    """
    return torch.einsum("ijk,krl,lmn->jrm", core1, core2, core3)


def contract_fixed_left(
    fixed_left_core: torch.Tensor,
    center_pair: torch.Tensor,
) -> torch.Tensor | None:
    """固定した左coreと2-site中心を縮約する。

    Args:
        fixed_left_core: shape (1, n1, r1) の固定core。
        center_pair: shape (r1, n2, n3, 1) の2-site中心。

    Returns:
        shape (n1, n2, n3) の全状態テンソル。
    """
    return torch.einsum("ijk,krls->jrl", fixed_left_core, center_pair)


B_old = merge_two_cores(G2, G3)
X_old_from_three = contract_three_cores(G1, G2, G3)
X_old_from_pair = (
    contract_fixed_left(G1, B_old) if B_old is not None else None
)

if all(value is not None for value in (B_old, X_old_from_three, X_old_from_pair)):
    b_old = B_old.reshape(-1)
    x_old = X_old_from_three.reshape(-1)
    assert B_old.shape == (1, 2, 2, 1)
    assert X_old_from_three.shape == X_old_from_pair.shape == (2, 2, 2)
    torch.testing.assert_close(X_old_from_three, X_old_from_pair, atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(
        B_old[0, :, :, 0],
        torch.tensor([[3.0, 4.0], [6.0, 8.0]], dtype=DTYPE),
        atol=ATOL,
        rtol=RTOL,
    )
    torch.testing.assert_close(
        x_old,
        torch.tensor([3.0, 4.0, 6.0, 8.0, 0.0, 0.0, 0.0, 0.0], dtype=DTYPE),
        atol=ATOL,
        rtol=RTOL,
    )
    print("stage 1: OK")
else:
    b_old = x_old = None
    print("stage 1: TODO")

stage 1: OK


## 4. 第2段階：固定coreから写像$P$を作る

### 目的

$$
x=Pb,\qquad P\in\mathbb R^{8\times4}
$$

を満たす$P$を、固定した$G_1$から構成します。

中心空間の単位ベクトル$e_k$を1つずつshape $(1,2,2,1)$へ戻し、`G1`と縮約して得た全状態を$P$の第$k$列にしてください。完成形を直接代入してはいけません。

$P$は固定coreと添字順序から決まり、現在の`B_old`の値には依存しません。

In [17]:
def build_projection_from_fixed_left(
    fixed_left_core: torch.Tensor,
    center_shape: tuple[int, int, int, int],
) -> torch.Tensor | None:
    """中心空間の基底を縮約し、中心から全状態への写像を作る。

    Args:
        fixed_left_core: shape (1, n1, r1) の固定core。
        center_shape: 2-site中心の (r1, n2, n3, 1)。

    Returns:
        各列が1つの中心基底に対応するshape (N, m) の写像行列。
    """
    center_dim = math.prod(center_shape)
    center_basis = torch.eye(
        center_dim, dtype=fixed_left_core.dtype, device=fixed_left_core.device
    )
    full_state_columns: list[torch.Tensor] = []

    for basis_index in range(center_dim):
        # TODO: 中心空間の単位ベクトルを1本取り出す。shape: (center_dim,)
        center_basis_vector = center_basis[basis_index]
        if center_basis_vector is None:
            return None

        # TODO: 単位ベクトルを2-site中心へ戻す。shape: center_shape
        center_basis_tensor = center_basis_vector.reshape(center_shape) 
        if center_basis_tensor is None:
            return None

        # TODO: 固定coreと中心基底を縮約する。shape: (n1, n2, n3)
        full_state_tensor =  contract_fixed_left(
            fixed_left_core, center_basis_tensor
        )
        if full_state_tensor is None:
            return None

        # TODO: 全状態を既定の添字順でベクトル化する。shape: (N,)
        full_state_vector = full_state_tensor.reshape(-1)
        if full_state_vector is None:
            return None
        full_state_columns.append(full_state_vector)

    # TODO: 各全状態ベクトルを列として結合する。shape: (N, center_dim)
    projection = torch.stack(full_state_columns, dim=1)
    if projection is None:
        return None
    return projection


P = build_projection_from_fixed_left(G1, (1, 2, 2, 1))

if P is not None and b_old is not None and x_old is not None:
    assert P.shape == (8, 4)
    torch.testing.assert_close(P @ b_old, x_old, atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(P.T @ P, torch.eye(4, dtype=DTYPE), atol=ATOL, rtol=RTOL)
    print("stage 2: OK")
else:
    print("stage 2: TODO")

stage 2: OK


## 5. 第3段階：有効Hamiltonianと更新前エネルギー

### 目的

第2・第3siteに作用する$h$から全Hamiltonian$H=I_2\otimes h$を作り、

$$
H_{\mathrm{eff}}=P^\top HP
$$

へ射影します。さらに、Rayleigh商

$$
E(z;A)=\frac{z^\top Az}{z^\top z}
$$

を実装します。

全状態から計算したエネルギーと、中心空間から計算したエネルギーが一致することを確認してください。

In [18]:
def build_full_hamiltonian(local_hamiltonian: torch.Tensor) -> torch.Tensor | None:
    """第1siteには恒等作用素を置き、全Hamiltonianを作る。

    Args:
        local_hamiltonian: 第2・第3siteに作用するshape (4, 4) の行列。

    Returns:
        shape (8, 8) の全Hamiltonian。
    """
    identity = torch.eye(
        2,
        dtype=local_hamiltonian.dtype,
        device=local_hamiltonian.device,
    )
    return torch.kron(identity, local_hamiltonian)


def project_hamiltonian(
    full_hamiltonian: torch.Tensor,
    projection: torch.Tensor,
) -> torch.Tensor | None:
    """全Hamiltonianを中心空間へ射影する。

    Args:
        full_hamiltonian: 全状態空間に作用するshape (N, N) の行列。
        projection: x = P bを表すshape (N, m) の写像行列。

    Returns:
        shape (m, m) の有効Hamiltonian。
    """
    return projection.T @ full_hamiltonian @ projection


def rayleigh_quotient(vector: torch.Tensor, operator: torch.Tensor) -> torch.Tensor | None:
    """非零ベクトルのRayleigh商を計算する。

    Args:
        vector: shape (d,) の非零実ベクトル。
        operator: shape (d, d) の実対称行列。

    Returns:
        スカラーTensorとしてのRayleigh商。
    """
    return (vector @ operator @ vector) / (vector @ vector)


H = build_full_hamiltonian(h)
H_eff = project_hamiltonian(H, P) if H is not None and P is not None else None
E_old_full = (
    rayleigh_quotient(x_old, H) if x_old is not None and H is not None else None
)
E_old_local = (
    rayleigh_quotient(b_old, H_eff)
    if b_old is not None and H_eff is not None
    else None
)

if all(value is not None for value in (H, H_eff, E_old_full, E_old_local)):
    assert H.shape == (8, 8)
    assert H_eff.shape == (4, 4)
    torch.testing.assert_close(H_eff, h, atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(E_old_full, E_old_local, atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(
        E_old_full,
        torch.tensor(-117.0 / 125.0, dtype=DTYPE),
        atol=ATOL,
        rtol=RTOL,
    )
    print("stage 3: OK")
else:
    print("stage 3: TODO")

stage 3: OK


## 6. 第4段階：固有値問題による2-site局所最適化

### 目的

`torch.linalg.eigh`で

$$
H_{\mathrm{eff}}b_{\mathrm{opt}}=E_{\mathrm{opt}}b_{\mathrm{opt}}
$$

の最小固有対を求めます。`b_opt`をshape $(1,2,2,1)$へ戻したものが`B_opt`です。

固有ベクトルの全体符号は不定です。要素を固定符号で比較せず、固有方程式の残差、ノルム、エネルギーで検証してください。ここが初めて状態とエネルギーを変更する段階です。

In [19]:
def lowest_eigenpair(
    effective_hamiltonian: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor] | None:
    """実対称な有効Hamiltonianの最小固有対を求める。

    Args:
        effective_hamiltonian: shape (m, m) の実対称行列。

    Returns:
        最小固有値のスカラーTensorと、正規化済み固有ベクトルのtuple。
    """
    # TODO: 実対称行列を固有値分解する。出力shape: (m,), (m, m)
    eigendecomposition = torch.linalg.eigh(effective_hamiltonian)
    if eigendecomposition is None:
        return None
    eigenvalues, eigenvectors = eigendecomposition

    # TODO: 最小固有値に対応する添字を選ぶ。出力: 0次元の整数Tensorまたはint
    lowest_index = torch.argmin(eigenvalues)
    if lowest_index is None:
        return None

    # TODO: 選んだ最小固有値を取り出す。出力shape: ()
    lowest_eigenvalue = eigenvalues[lowest_index]
    if lowest_eigenvalue is None:
        return None

    # TODO: 同じ添字の固有ベクトルを列から取り出す。出力shape: (m,)
    lowest_eigenvector = eigenvectors[:, lowest_index]
    if lowest_eigenvector is None:
        return None
    return lowest_eigenvalue, lowest_eigenvector


lowest_solution = lowest_eigenpair(H_eff) if H_eff is not None else None

if lowest_solution is not None and H_eff is not None and P is not None and H is not None:
    E_opt, b_opt = lowest_solution
    B_opt = b_opt.reshape(1, 2, 2, 1)
    x_opt = P @ b_opt
    E_opt_full = rayleigh_quotient(x_opt, H)
    residual = torch.linalg.vector_norm(H_eff @ b_opt - E_opt * b_opt)
    if E_opt_full is not None:
        assert B_opt.shape == (1, 2, 2, 1)
        torch.testing.assert_close(
            E_opt, torch.tensor(-2.0, dtype=DTYPE), atol=ATOL, rtol=RTOL
        )
        torch.testing.assert_close(
            torch.linalg.vector_norm(b_opt),
            torch.tensor(1.0, dtype=DTYPE),
            atol=ATOL,
            rtol=RTOL,
        )
        assert residual.item() <= ATOL
        torch.testing.assert_close(E_opt_full, E_opt, atol=ATOL, rtol=RTOL)
        print("stage 4: OK")
    else:
        print("stage 4: SKIP (rayleigh_quotient is TODO)")
else:
    E_opt = b_opt = B_opt = x_opt = E_opt_full = None
    print("stage 4: TODO")


stage 4: OK


## 7. 第5段階：打ち切りなしのSVD分割

### 目的

`B_opt`を

$$
M_{\mathrm{opt}}\in\mathbb R^{(r_1n_2)\times n_3}
=\mathbb R^{2\times2}
$$

へ行列化し、共通処理として一度だけSVDします。`torch.linalg.svd`の第3出力は`Vh`です。

右へ進む配置として、

$$
G_{2,\mathrm{right}}\leftarrow U,\qquad
G_{3,\mathrm{right}}\leftarrow\Sigma V^\top
$$

を作ります。SVDそのものと特異値の配置を別関数にし、第6段階の左向き配置と同じ関数構成で比較します。特異値が縮退しているため、`U`や`Vh`が単位行列になることは要求しません。

In [20]:
def factorize_two_site_center(
    optimized_pair: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor] | None:
    """2-site中心を行列化し、打ち切らずにSVDする。

    Args:
        optimized_pair: shape (r1, n2, n3, 1) の最適化済み中心。

    Returns:
        U、特異値S、Vhのtuple。
    """
    r_left, n_left, n_right, r_right = optimized_pair.shape

    # TODO: 左bondと左物理添字、右物理添字と右bondをまとめる。
    # 入力shape: (r_left, n_left, n_right, r_right)
    # 出力shape: (r_left * n_left, n_right * r_right)
    pair_matrix = optimized_pair.reshape(r_left * n_left, n_right * r_right)
    if pair_matrix is None:
        return None

    # TODO: pair_matrixを打ち切らずにSVDする。
    # 出力shape: U=(rows, chi), S=(chi,), Vh=(chi, columns)
    svd_result = torch.linalg.svd(pair_matrix, full_matrices=False)
    if svd_result is None:
        return None
    return svd_result


def place_singular_values_for_right_sweep(
    U: torch.Tensor,
    singular_values: torch.Tensor,
    Vh: torch.Tensor,
    optimized_pair: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor] | None:
    """SVD因子を右向きsweep用のcore配置へ変換する。

    Args:
        U: SVDの左特異ベクトル行列。
        singular_values: SVDの特異値ベクトル。
        Vh: SVDの右特異ベクトルを行に持つ行列。
        optimized_pair: 最適化済み2-site中心。

    Returns:
        左正準なG2_rightとG3_rightのtuple。
    """
    r_left, n_left, n_right, r_right = optimized_pair.shape
    bond_dimension = singular_values.numel()

    # TODO: 各特異値をVhの対応する行へ掛ける。
    # 出力shape: (bond_dimension, n_right * r_right)
    weighted_Vh = torch.diag(singular_values) @ Vh
    if weighted_Vh is None:
        return None

    # TODO: Uを左coreへ戻す。shape: (r_left, n_left, bond_dimension)
    G2_right = U.reshape(r_left, n_left, bond_dimension)
    if G2_right is None:
        return None

    # TODO: 重み付きVhを右coreへ戻す。shape: (bond_dimension, n_right, r_right)
    G3_right = weighted_Vh.reshape(bond_dimension, n_right, r_right)
    if G3_right is None:
        return None
    return G2_right, G3_right


svd_factors = factorize_two_site_center(B_opt) if B_opt is not None else None

if svd_factors is not None and B_opt is not None:
    U, S, Vh = svd_factors
    right_split = place_singular_values_for_right_sweep(U, S, Vh, B_opt)
else:
    U = S = Vh = right_split = None

if right_split is not None and B_opt is not None and H is not None:
    G2_right, G3_right = right_split
    B_reconstructed_right = merge_two_cores(G2_right, G3_right)
    X_reconstructed_right = (
        contract_fixed_left(G1, B_reconstructed_right)
        if B_reconstructed_right is not None
        else None
    )
    E_after_exact_split = (
        rayleigh_quotient(X_reconstructed_right.reshape(-1), H)
        if X_reconstructed_right is not None
        else None
    )
    if all(
        value is not None
        for value in (B_reconstructed_right, X_reconstructed_right, E_after_exact_split)
    ):
        assert G2_right.shape == (1, 2, 2)
        assert G3_right.shape == (2, 2, 1)
        torch.testing.assert_close(B_reconstructed_right, B_opt, atol=ATOL, rtol=RTOL)
        torch.testing.assert_close(
            E_after_exact_split,
            torch.tensor(-2.0, dtype=DTYPE),
            atol=ATOL,
            rtol=RTOL,
        )
        G2_matrix = G2_right.reshape(2, 2)
        torch.testing.assert_close(
            G2_matrix.T @ G2_matrix,
            torch.eye(2, dtype=DTYPE),
            atol=ATOL,
            rtol=RTOL,
        )
        print("stage 5: OK")
    else:
        print("stage 5: SKIP (a dependency is TODO)")
else:
    G2_right = G3_right = None
    B_reconstructed_right = X_reconstructed_right = E_after_exact_split = None
    print("stage 5: TODO")


stage 5: OK


## 8. 第6段階：特異値の配置と右正準形

### 目的

第5段階の共通SVDから、右向き配置関数と同じ引数・戻り値を持つ関数で、左へ進む配置

$$
G_{2,\mathrm{left}}\leftarrow U\Sigma,\qquad
G_{3,\mathrm{left}}\leftarrow V^\top
$$

を作ります。

この操作は第3coreを最適化し直すものではありません。同じ`B_opt`を保ったまま、第3coreを右正準形にし、orthogonality centerを左側へ配置する操作です。

In [21]:
def place_singular_values_for_left_sweep(
    U: torch.Tensor,
    singular_values: torch.Tensor,
    Vh: torch.Tensor,
    optimized_pair: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor] | None:
    """同じSVDを左向きsweep用のcore配置へ変換する。

    Args:
        U: SVDの左特異ベクトル行列。
        singular_values: SVDの特異値ベクトル。
        Vh: SVDの右特異ベクトルを行に持つ行列。
        optimized_pair: 最適化済み2-site中心。

    Returns:
        G2_leftと右正準なG3_leftのtuple。
    """
    r_left, n_left, n_right, r_right = optimized_pair.shape
    bond_dimension = singular_values.numel()

    # TODO: Uの各列へ対応する特異値を掛ける。
    # 入力shape: U=(r_left * n_left, bond_dimension), S=(bond_dimension,)
    # 出力shape: (r_left * n_left, bond_dimension)
    weighted_U = U @ torch.diag(singular_values)
    if weighted_U is None:
        return None

    # TODO: 重み付きUを左coreへ戻す。shape: (r_left, n_left, bond_dimension)
    G2_left = weighted_U.reshape(r_left, n_left, bond_dimension)
    if G2_left is None:
        return None

    # TODO: Vhを右coreへ戻す。shape: (bond_dimension, n_right, r_right)
    G3_left = Vh.reshape(bond_dimension, n_right, r_right)
    if G3_left is None:
        return None
    return G2_left, G3_left


left_split = (
    place_singular_values_for_left_sweep(U, S, Vh, B_opt)
    if all(value is not None for value in (U, S, Vh, B_opt))
    else None
)

if left_split is not None and B_opt is not None and H is not None:
    G2_left, G3_left = left_split
    B_reconstructed_left = merge_two_cores(G2_left, G3_left)
    X_reconstructed_left = (
        contract_fixed_left(G1, B_reconstructed_left)
        if B_reconstructed_left is not None
        else None
    )
    E_after_center_move = (
        rayleigh_quotient(X_reconstructed_left.reshape(-1), H)
        if X_reconstructed_left is not None
        else None
    )
    if all(
        value is not None
        for value in (B_reconstructed_left, X_reconstructed_left, E_after_center_move)
    ):
        assert G2_left.shape == (1, 2, 2)
        assert G3_left.shape == (2, 2, 1)
        torch.testing.assert_close(B_reconstructed_left, B_opt, atol=ATOL, rtol=RTOL)
        G3_matrix = G3_left.reshape(2, 2)
        torch.testing.assert_close(
            G3_matrix @ G3_matrix.T,
            torch.eye(2, dtype=DTYPE),
            atol=ATOL,
            rtol=RTOL,
        )
        torch.testing.assert_close(
            E_after_center_move,
            torch.tensor(-2.0, dtype=DTYPE),
            atol=ATOL,
            rtol=RTOL,
        )
        print("stage 6: OK")
    else:
        print("stage 6: SKIP (a dependency is TODO)")
else:
    G2_left = G3_left = None
    B_reconstructed_left = X_reconstructed_left = E_after_center_move = None
    print("stage 6: TODO")


stage 6: OK


## 9. 第7段階：bond dimension 1への打ち切り

### 目的

PyTorchが返したSVDから特異値を1つだけ残し、rank 1近似`M_trunc`を作ります。再正規化前に

$$
\|M_{\mathrm{opt}}-M_{\mathrm{trunc}}\|_F^2
=\sigma_2^2=\frac12
$$

を確認してください。

今回は特異値が縮退しているため、最良rank 1近似は一意ではありません。自動SVDの結果についてエネルギーが必ず$-1$になるというassertは置きません。

その後、手計算との対応を確認するため、全体符号をそろえた

$$
M_{\mathrm{ref}}=\frac1{\sqrt2}I_2
$$

から、第1方向を残すrank 1近似を別に作ります。自動SVDの出力と、方向を明示した参照計算を混同しないでください。

In [22]:
def truncate_first_svd_component(
    U: torch.Tensor,
    singular_values: torch.Tensor,
    Vh: torch.Tensor,
) -> torch.Tensor | None:
    """PyTorchが返したSVDの第1成分だけで行列を再構成する。

    Args:
        U: SVDの左特異ベクトル行列。
        singular_values: 降順の特異値ベクトル。
        Vh: SVDの右特異ベクトルを行に持つ行列。

    Returns:
        第1特異成分だけを残したrank 1行列。
    """
    # TODO: Uの第1列を列ベクトルとして取り出す。shape: (rows, 1)
    first_left_singular_vector = U[:, :1]
    if first_left_singular_vector is None:
        return None

    # TODO: 先頭の特異値を取り出す。shape: ()
    first_singular_value = singular_values[0]
    if first_singular_value is None:
        return None

    # TODO: Vhの第1行を行ベクトルとして取り出す。shape: (1, columns)
    first_right_singular_vector = Vh[:1, :]
    if first_right_singular_vector is None:
        return None

    # TODO: 3要素を掛けて第1特異成分を再構成する。shape: (rows, columns)
    rank_one_matrix = (
        first_singular_value
        * (first_left_singular_vector @ first_right_singular_vector)
    )
    if rank_one_matrix is None:
        return None
    return rank_one_matrix


def build_reference_rank_one_truncation(
    reference_matrix: torch.Tensor,
) -> torch.Tensor | None:
    """指定された第1座標方向を残す参照rank 1近似を作る。

    Args:
        reference_matrix: (1 / sqrt(2)) I2であるshape (2, 2) の参照行列。

    Returns:
        第1座標方向だけを残したshape (2, 2) の行列。
    """
    truncated = torch.zeros_like(reference_matrix)
    truncated[0, 0] = reference_matrix[0, 0]
    return truncated


M_opt = B_opt.reshape(2, 2) if B_opt is not None else None
M_trunc = (
    truncate_first_svd_component(U, S, Vh)
    if all(value is not None for value in (U, S, Vh))
    else None
)

if M_opt is not None and M_trunc is not None and S is not None:
    truncation_error_sq = torch.linalg.matrix_norm(M_opt - M_trunc).square()
    torch.testing.assert_close(truncation_error_sq, S[1].square(), atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(
        truncation_error_sq, torch.tensor(0.5, dtype=DTYPE), atol=ATOL, rtol=RTOL
    )
    b_trunc = M_trunc.reshape(-1)
    b_trunc_normalized = b_trunc / torch.linalg.vector_norm(b_trunc)
    E_trunc_automatic = rayleigh_quotient(b_trunc_normalized, h)
    print("automatic truncation check: OK")
    if E_trunc_automatic is not None:
        print(
            "automatic-SVD rank-1 energy (diagnostic only):",
            E_trunc_automatic.item(),
        )
    else:
        print("automatic-SVD energy: SKIP (rayleigh_quotient is TODO)")
else:
    truncation_error_sq = b_trunc = b_trunc_normalized = E_trunc_automatic = None
    print("automatic truncation: TODO")

M_ref = torch.eye(2, dtype=DTYPE) / math.sqrt(2.0)
M_trunc_ref = build_reference_rank_one_truncation(M_ref)

if M_trunc_ref is not None:
    b_trunc_ref = M_trunc_ref.reshape(-1)
    b_trunc_ref = b_trunc_ref / torch.linalg.vector_norm(b_trunc_ref)
    E_trunc_ref = rayleigh_quotient(b_trunc_ref, h)
    assert M_trunc_ref.shape == M_ref.shape
    assert torch.linalg.matrix_rank(M_trunc_ref).item() == 1
    torch.testing.assert_close(
        M_trunc_ref[0, 0],
        M_ref[0, 0],
        atol=ATOL,
        rtol=RTOL,
    )
    if E_trunc_ref is not None:
        torch.testing.assert_close(
            E_trunc_ref,
            torch.tensor(-1.0, dtype=DTYPE),
            atol=ATOL,
            rtol=RTOL,
        )
        print("reference-direction truncation: OK")
    else:
        print("reference-direction energy: SKIP (rayleigh_quotient is TODO)")
else:
    b_trunc_ref = E_trunc_ref = None
    print("reference-direction truncation: TODO")


automatic truncation check: OK
automatic-SVD rank-1 energy (diagnostic only): -1.0
reference-direction truncation: OK


## 10. 任意の発展問題：固定coreが変わると$P$はどう変わるか

この節は**任意**です。第1〜第7段階の完了条件には含めません。本文で繰り返し確認した「$P$は固定coreから決まる」を、別の固定coreで検証します。

$$
(G_1^{\mathrm{general}})_{1,:,1}=\left(\frac35,\frac45\right)
$$

として、第2段階で実装した関数から`P_general`を作ってください。

確認すること：

- `P_general`は$B$の現在値に依存せず、`G1_general`だけから作られる。
- $G1_general$は正規化されているので$P_{\mathrm{general}}^\top P_{\mathrm{general}}=I_4$。
- `H = I_2 \otimes h`という今回の特別なHamiltonianでは、対応する有効Hamiltonianも`h`になる。

次に固定coreを$(3,4)$へ変え、$P^\top P=25I_4$になることを確認してください。正準化されていない場合、分母を無条件に$b^\top b$へ置き換えられない理由を説明してください。

In [23]:
G1_general = torch.tensor([[[3.0 / 5.0], [4.0 / 5.0]]], dtype=DTYPE)
G1_unnormalized = torch.tensor([[[3.0], [4.0]]], dtype=DTYPE)

P_general = build_projection_from_fixed_left(G1_general, (1, 2, 2, 1))
P_unnormalized = build_projection_from_fixed_left(G1_unnormalized, (1, 2, 2, 1))

H_eff_general = (
    project_hamiltonian(H, P_general)
    if H is not None and P_general is not None
    else None
)

if all(
    value is not None
    for value in (P_general, P_unnormalized, H, H_eff_general)
):
    identity4 = torch.eye(4, dtype=DTYPE)
    torch.testing.assert_close(P_general.T @ P_general, identity4, atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(H_eff_general, h, atol=ATOL, rtol=RTOL)
    torch.testing.assert_close(
        P_unnormalized.T @ P_unnormalized, 25.0 * identity4, atol=ATOL, rtol=RTOL
    )
    print("optional projection problem: OK")
else:
    print("optional projection problem: SKIP")

optional projection problem: OK


## 11. 最後の整理問題

コードの結果を根拠に、次を短く説明してください。完成文は用意していません。

1. 結合、局所最適化、打ち切りなしのSVD分割、特異値の配置変更、rank 1打ち切りのうち、どの操作で状態が変わったか。
2. エネルギーを$-0.936$から$-2$へ下げた操作は何か。
3. 打ち切りなしのSVD分割で、bond dimensionが1から2へ増えた理由。
4. SVD打ち切りでエネルギーが$-2$から上がり得る理由。
5. $P$を作る際、中心の現在値ではなく中心空間の単位ベクトルを使う理由。
6. ベクトル化の順序を中心側と全状態側で一貫させる必要がある理由。
7. このNotebookを完了しても、左右へ往復するsweep全体を実装したことにならない理由。

### 数値の整理

- 更新前：$E=-117/125=-0.936$。
- 局所最適化後：$E=-2$。
- 打ち切りなしの分割後：$E=-2$。
- 明示した第1座標方向を残すrank 1打ち切り後：$E=-1$。

今回のHamiltonianは第1siteに作用せず、打ち切りなしなら1回の局所更新で基底状態へ到達する特別な例です。繰り返しsweepによる改善を確認したとは扱いません。